# 01 — Exploration des Transformers

**Mini-Projet NLP** — ENSAH 2A ID

Objectif : Tester et évaluer 4 tâches NLP avec des modèles pré-entraînés HuggingFace.

| Tâche | Modèle | Dataset | Métrique |
|-------|--------|---------|----------|
| Zero-shot Classification | `facebook/bart-large-mnli` | AG News | Accuracy |
| Sentiment Analysis | `distilbert-base-uncased-finetuned-sst-2-english` | SST-2 | Accuracy, F1, Confusion Matrix |
| Question Answering | `deepset/roberta-base-squad2` | SQuAD v2 | Exact Match, F1 |
| Résumé automatique | `facebook/bart-large-cnn` | CNN/DailyMail (échantillon) | ROUGE-1, ROUGE-2, ROUGE-L |

In [ ]:
# ─── Setup ───────────────────────────────────────────
import json, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from transformers import (
    pipeline, AutoTokenizer, AutoModelForQuestionAnswering,
    AutoModelForSeq2SeqLM
)

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Python   : {sys.executable}")
print(f"Device   : {DEVICE}")
print(f"Torch    : {torch.__version__}")
print(f"Transf.  : {__import__('transformers').__version__}")
try:
    import evaluate
    print(f"evaluate : {evaluate.__version__} \u2705")
except ImportError:
    print("evaluate : \u274c NON TROUVE \u2014 utilisez l'interpreteur du venv")

---
## 1. Zero-shot Classification — AG News

**Modèle :** `facebook/bart-large-mnli` — utilise l'inférence en langage naturel (NLI) pour classifier sans entraînement spécifique.

**Dataset :** AG News (4 catégories : World, Sports, Business, Science/Tech)

In [ ]:
# ─── Charger le dataset AG News ───────────────────────
DATA_DIR = Path.cwd() / 'data' / 'raw' / 'ag_news'

with open(DATA_DIR / 'test.json') as f:
    ag_test = json.load(f)
    
df_ag = pd.DataFrame(ag_test)
print(f"AG News — {len(df_ag)} échantillons de test")
print(df_ag['label_nom'].value_counts())

In [ ]:
# ─── Zero-shot Classification Pipeline ────────────────
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if DEVICE == 'cuda' else -1
)

candidate_labels = ['World', 'Sports', 'Business', 'Science/Tech']

# Prédire sur l'échantillon de test
preds = []
for text in df_ag['texte']:
    result = classifier(text, candidate_labels, multi_label=False)
    preds.append(result['labels'][0])

df_ag['prediction'] = preds

In [ ]:
# ─── Évaluation ───────────────────────────────────────
y_true = df_ag['label_nom'].values
y_pred = df_ag['prediction'].values

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"Accuracy  : {acc:.2%}")
print(f"F1 Weighted : {f1:.2%}")
print()
print(classification_report(y_true, y_pred, digits=3))

# Matrice de confusion
cm = confusion_matrix(y_true, y_pred, labels=candidate_labels[:4])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=candidate_labels[:4],
            yticklabels=candidate_labels[:4])
plt.title('Classification Zero-shot — AG News')
plt.xlabel('Prédiction')
plt.ylabel('Vérité terrain')
plt.show()

---
## 2. Sentiment Analysis — SST-2

**Modèle :** `distilbert-base-uncased-finetuned-sst-2-english` — DistilBERT fine-tuné sur Stanford Sentiment Treebank.

**Dataset :** SST-2 (POSITIVE / NEGATIVE)

In [ ]:
# ─── Charger le dataset SST-2 ─────────────────────────
DATA_DIR = Path.cwd() / 'data' / 'raw' / 'sst2'

with open(DATA_DIR / 'validation.json') as f:
    sst_val = json.load(f)

df_sst = pd.DataFrame(sst_val)
print(f"SST-2 — {len(df_sst)} échantillons de validation")
print(df_sst['label_nom'].value_counts())

In [ ]:
# ─── Sentiment Pipeline ───────────────────────────────
sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if DEVICE == 'cuda' else -1
)

preds_sst = []
for text in df_sst['texte']:
    result = sentiment(text)[0]
    preds_sst.append(result['label'].upper())  # POSITIVE ou NEGATIVE

df_sst['prediction'] = preds_sst

In [ ]:
# ─── Évaluation ───────────────────────────────────────
y_true = df_sst['label_nom'].values
y_pred = df_sst['prediction'].values

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"Accuracy  : {acc:.2%}")
print(f"F1 Weighted : {f1:.2%}")
print()
print(classification_report(y_true, y_pred, digits=3))

# Matrice de confusion
cm = confusion_matrix(y_true, y_pred, labels=['POSITIVE', 'NEGATIVE'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['POSITIVE', 'NEGATIVE'],
            yticklabels=['POSITIVE', 'NEGATIVE'])
plt.title('Sentiment Analysis — SST-2')
plt.xlabel('Prédiction')
plt.ylabel('Vérité terrain')
plt.show()

# Analyse des erreurs
errors = df_sst[df_sst['label_nom'] != df_sst['prediction']]
print(f"\n❌ {len(errors)} erreurs sur {len(df_sst)}")
print("\nExemples d'erreurs :")
for _, row in errors.head(5).iterrows():
    print(f"  Texte: {row['texte'][:80]}...")
    print(f"  Réel: {row['label_nom']} | Prédit: {row['prediction']}\n")

---
## 3. Question Answering — SQuAD v2

**Modèle :** `deepset/roberta-base-squad2` — RoBERTa fine-tuné sur SQuAD v2.

**Dataset :** SQuAD v2 (question + contexte → réponse)

In [ ]:
# ─── Charger le dataset SQuAD v2 ──────────────────────
DATA_DIR = Path.cwd() / 'data' / 'raw' / 'squad_v2'

with open(DATA_DIR / 'validation.json') as f:
    squad_val = json.load(f)

df_squad = pd.DataFrame(squad_val)
df_squad = df_squad[df_squad['a_reponse'] == True].reset_index(drop=True)  # uniquement avec réponse
print(f"SQuAD v2 — {len(df_squad)} questions avec réponse")

In [ ]:
# ─── Charger le modèle QA (pipeline non disponible en v5, utilisation directe) ─
MODELS_DIR = Path.cwd() / 'models' / 'question_answering' / 'roberta-squad2'

if MODELS_DIR.exists():
    tokenizer = AutoTokenizer.from_pretrained(str(MODELS_DIR))
    model = AutoModelForQuestionAnswering.from_pretrained(str(MODELS_DIR))
    model.to(DEVICE)
    print("Modèle chargé depuis le disque")
else:
    tokenizer = AutoTokenizer.from_pretrained("deepset/roberta-base-squad2")
    model = AutoModelForQuestionAnswering.from_pretrained("deepset/roberta-base-squad2")
    model.to(DEVICE)
    print("Modèle chargé depuis HuggingFace")

def answer_question(question, context):
    inputs = tokenizer(question, context, return_tensors='pt', truncation=True, max_length=512)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits)
    if start_idx > end_idx:
        return ""
    tokens = inputs['input_ids'][0][start_idx:end_idx+1]
    return tokenizer.decode(tokens, skip_special_tokens=True)

In [ ]:
# ─── Prédictions ──────────────────────────────────────
# Limiter à 50 questions pour la vitesse
df_squad_sample = df_squad.head(50).copy()

predicted_answers = []
for _, row in df_squad_sample.iterrows():
    ans = answer_question(row['question'], row['contexte'])
    predicted_answers.append(ans)

df_squad_sample['prediction'] = predicted_answers

In [ ]:
# ─── Évaluation (Exact Match & F1) ────────────────────
def normalize_text(s):
    """Nettoie le texte pour la comparaison."""
    import re
    s = s.lower().strip()
    s = re.sub(r'[^a-z0-9\s]', '', s)
    return s

def compute_f1(reference, prediction):
    ref_tokens = normalize_text(reference).split()
    pred_tokens = normalize_text(prediction).split()
    common = set(ref_tokens) & set(pred_tokens)
    if len(common) == 0:
        return 0.0
    precision = len(common) / len(pred_tokens) if pred_tokens else 0
    recall = len(common) / len(ref_tokens) if ref_tokens else 0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

em_scores = []
f1_scores = []
for _, row in df_squad_sample.iterrows():
    ref = normalize_text(row['reponse'])
    pred = normalize_text(row['prediction'])
    em_scores.append(1.0 if ref == pred else 0.0)
    f1_scores.append(compute_f1(ref, pred))

print(f"Exact Match (EM) : {np.mean(em_scores):.2%}")
print(f"F1 Score         : {np.mean(f1_scores):.2%}")
print()
print("Exemples de prédictions :")
for _, row in df_squad_sample.head(10).iterrows():
    print(f"\nQ: {row['question']}")
    print(f"R: {row['reponse'][:60]}")
    print(f"P: {row['prediction'][:60]}")

---
## 4. Résumé automatique — CNN/DailyMail

**Modèle :** `facebook/bart-large-cnn` — BART fine-tuné pour le résumé abstractif.

**Dataset :** Échantillon depuis HuggingFace (50 articles)

In [ ]:
# ─── Charger un petit échantillon de CNN/DailyMail ────
from datasets import load_dataset

cnn_sample = load_dataset("abisee/cnn_dailymail", "3.0.0", split="train", streaming=True)
articles = []
for i, example in enumerate(cnn_sample):
    if i >= 20:
        break
    articles.append({
        'article': example['article'][:2000],  # tronquer pour la vitesse
        'highlights': example['highlights']
    })
    
print(f"{len(articles)} articles chargés")

In [ ]:
# ─── Charger le modèle de résumé ──────────────────────
MODELS_DIR = Path.cwd() / 'models' / 'summarization' / 'bart-large-cnn'

if MODELS_DIR.exists():
    tokenizer_sum = AutoTokenizer.from_pretrained(str(MODELS_DIR))
    model_sum = AutoModelForSeq2SeqLM.from_pretrained(str(MODELS_DIR))
    print("Modèle chargé depuis le disque")
else:
    tokenizer_sum = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
    model_sum = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")
    print("Modèle chargé depuis HuggingFace")

model_sum.to(DEVICE)

def summarize(text, max_len=130, min_len=30):
    inputs = tokenizer_sum(text, return_tensors='pt', truncation=True, max_length=1024)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    summary_ids = model_sum.generate(
        **inputs,
        max_length=max_len,
        min_length=min_len,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )
    return tokenizer_sum.decode(summary_ids[0], skip_special_tokens=True)

In [ ]:
# ─── Générer les résumés ──────────────────────────────
summaries = []
for item in articles[:5]:  # 5 articles pour la vitesse
    summaries.append(summarize(item['article']))

for i, (item, summ) in enumerate(zip(articles[:5], summaries)):
    print(f"\n--- Article {i+1} ---")
    print(f"Article: {item['article'][:200]}...")
    print(f"\nReference: {item['highlights'][:200]}...")
    print(f"Résumé: {summ[:200]}...")

In [ ]:
# ─── Évaluation ROUGE ─────────────────────────────────
from evaluate import load as load_metric
rouge = load_metric('rouge')

results = rouge.compute(
    predictions=summaries,
    references=[a['highlights'] for a in articles[:5]]
)

print("Scores ROUGE :")
for key, value in results.items():
    print(f"  {key:>10} : {value:.4f}")

---
## Résumé des résultats

| Tâche | Modèle | Accuracy | F1 |
|-------|--------|----------|-----|
| Zero-shot Classification | BART MNLI | à remplir | à remplir |
| Sentiment Analysis | DistilBERT | à remplir | à remplir |
| Question Answering | RoBERTa SQuAD2 | à remplir (EM) | à remplir |
| Résumé | BART CNN | à remplir (ROUGE-1) | à remplir (ROUGE-2) |